# 🖼️ vLLM 多模态与工具调用 — 实战深度解析

**本文目标**：深入理解 vLLM 如何支持多模态模型和工具调用，从数据流到显存管理。

读完这篇你会理解：
- vLLM 的 MultiModalRegistry 架构
- Visual tokens 在 PagedAttention 中的处理
- 工具调用在 vLLM 中的完整生命周期
- 生产环境的最佳配置

## 1. 多模态架构

### 1.1 MultiModalRegistry 设计

```python
# vLLM 的多模态架构 (简化)

class MultiModalRegistry:
    """注册和管理所有的多模态模型类型"""
    
    _registry = {
        "llava": LLaVAProcessor,
        "llava-next": LLaVANextProcessor,
        "phi3-vision": Phi3VisionProcessor,
        "internvl": InternVLProcessor,
        "qwen-vl": QwenVLProcessor,
        "fuyu": FuyuProcessor,
        # ... 持续增加
    }

class MultiModalProcessor:
    """通用的多模态处理接口"""
    
    def process_image(self, image: PIL.Image) -> MultiModalData:
        """图片 → visual embeddings"""
        # 1. 预处理 (resize, normalize)
        # 2. Visual Encoder 前向传播
        # 3. Projector (对齐到 LLM 维度)
        # 4. 返回 visual embeddings + metadata
    
    def get_num_visual_tokens(self, image_size) -> int:
        """估算 visual token 数量"""
        # LLaVA: (H/14) * (W/14) → 576 for 1024x1024
        # InternVL: dynamic resolution → 可变
```

### 1.2 Visual Tokens 在推理中的流程

```python
# 多模态请求的完整处理
class MultiModalSequence:
    
    def process(self, request: ChatCompletionRequest):
        # 1. 提取图片
        images = extract_images(request.messages)
        
        # 2. Visual Encoder (可能与其他请求共享)
        visual_embeds = []
        for img in images:
            embed = self.visual_processor.process_image(img)
            visual_embeds.append(embed)
        
        # 3. 构建 prompt embedding
        # [<image> token placeholders] + [text tokens]
        prompt = self.build_multimodal_prompt(request.messages, visual_embeds)
        
        # 4. LLM prefill
        # visual tokens 和 text tokens 拼接 → 一起做 attention
        # visual tokens 的 KV Cache → 存入 PagedAttention blocks
        
        # 5. Decode (正常流程)
        # 后续生成的 text tokens 对 visual tokens 做 cross-attention
        # (实际上是 self-attention, 因为 visual tokens 的 K,V 在 block table 里)
```

### 1.3 显存管理等式

```
多模态请求的显存公式:

总显存 = LLM Weights + Visual Encoder + Visual KV Cache + Text KV Cache + Misc

Visual Encoder (常驻, 可 offload):
  ViT-L/CLIP: ~1.8 GB (FP16)
  使用后可以 offload → 但会增加下次图片处理的延迟

Visual KV Cache:
  per_image = num_visual_tokens × 2 × n_layers × n_kv_heads × head_dim × dtype
  LLaVA-7B, 1 张 1024x1024 图:
    576 tokens × 2 × 32 × 32 × 128 × 2 bytes = ~72 MB
  10 并发 × 5 张图:
    50 × 72 MB = 3.6 GB

Text KV Cache:
  同上, 但文本通常短 → ~0.5-2 GB per request

Visual Encoder 的共享:
  同一张图被多个请求引用 → visual embeddings 可以缓存
  但 KV Cache 不会自动共享 (除非启用 APC + 图片相同)
  vLLM 当前版本的 visual prefix caching 有限
  → 这是 SGLang 的优势领域
```

## 2. 工具调用 (Function Calling)

### 2.1 vLLM 的工具调用流程

```
┌─────────────────────────────────────────────────────────┐
│                vLLM 工具调用生命周期                       │
├─────────────────────────────────────────────────────────┤
│                                                          │
│  1. Chat Template 处理                                    │
│     messages = [                                         │
│       {"role": "system", "content": "You have tools..."},│
│       {"role": "user", "content": "北京天气?"}            │
│     ]                                                    │
│     → 应用 chat template → tokenize → input_ids          │
│                                                          │
│  2. 模型推理                                              │
│     → 生成: '{"name": "get_weather", "parameters": ...}' │
│                                                          │
│  3. 解析 tool_call                                        │
│     vLLM 可以自动检测并解析 JSON tool_call               │
│     → tool_call = {"name": "get_weather", "args": {...}} │
│                                                          │
│  4. 执行工具 (application 层)                              │
│     result = execute_tool(tool_call)  ← 你的代码          │
│                                                          │
│  5. 注入 tool_result                                      │
│     messages.append({                                     │
│       "role": "tool",                                    │
│       "tool_call_id": "...",                              │
│       "content": json.dumps(result)                      │
│     })                                                    │
│     → 重新 tokenize + prefill tool_result                │
│     → KV Cache 追加到现有 Sequence                            │
│                                                          │
│  6. 继续推理                                              │
│     → 基于 tool_result 生成最终回复                       │
│                                                          │
└─────────────────────────────────────────────────────────┘
```

### 2.2 工具调用的显存管理

```python
# 关键: tool_result 注入时 KV Cache 的处理

# vLLM 的优化:
# tool_result 注入 = 追加新的 prompt tokens
# → Scheduler 将请求重新当作 prefill 处理
# → Block Manager 为该请求追加分配新的 KV Cache blocks
# → 但之前的 KV Cache (system prompt + 之前轮次) 保持不变!

# 这意味着:
# 请求的 KV Cache = system_prompt_blocks + round1_blocks + round2_blocks + ...
# 每轮追加, 永不释放 (直到请求结束)

# 工具调用的 KV Cache 管理策略:
# 1. 限制最大轮数 (application 层控制)
# 2. 工具结果截断 (max 2000 tokens)
# 3. 考虑滑动窗口: 旧轮次的 tokens 移出 KV Cache
#    (vLLM 原生不支持滑动窗口 KV Cache, 需要 truncate + recompute)
```

### 2.3 最佳实践配置

```bash
# vLLM 工具调用优化配置
vllm serve model     --enable-prefix-caching \         # tool call 格式可被缓存
    --max-model-len 16384 \           # 给多轮留足空间
    --max-num-seqs 16 \               # 降低并发, 每请求更多 KV Cache
    --gpu-memory-utilization 0.90     --enable-auto-tool-choice \       # 自动检测 tool_call JSON
    --tool-call-parser "llama3_json"  # 使用模型特定的 parser

# 对于长 Agent 会话, 建议:
# 1. 在应用中限制 max_rounds ≤ 10
# 2. tool_result 截断到 2000 tokens
# 3. 监控 request-level KV Cache 大小
# 4. 考虑分离 Agent 服务和纯文本服务
```

## 3. 混合负载调度

### 3.1 多模态和工具调用对调度的冲击

```
纯文本请求:
  Prefill: ~500 tokens → ~50ms
  Decode: ~100 steps × 20ms → ~2s
  → 可预测, 低延迟

多模态请求:
  Prefill: ~3,000 tokens (图) + 50 tokens (文) → ~300ms
  Decode: ~100 steps × 20ms → ~2s
  → Prefill 是纯文本的 6x!

工具调用请求:
  Prefill 1: system prompt ~2,000 tokens → ~200ms
  Prefill 2: tool_result注入 ~1,000 tokens → ~100ms
  Prefill 3: tool_result注入 ~3,000 tokens → ~300ms
  ...
  → 多次 prefill, 每次都可能阻塞其他请求

vLLM 的应对:
  --max-num-batched-tokens 控制每次 prefill 的最大 token 数
  → 长的 prefill 被拆分
  → 但增加了总的 TTFT
```

### 3.2 分离部署 vs 混合部署

| 策略 | 优点 | 缺点 |
|------|------|------|
| **分离部署** | 每种请求类型独立优化 | 需要管理多个服务 |
| | 纯文本: 高并发低延迟 | 增加运维成本 |
| | Agent: 大 KV Cache, prefix caching | |
| | 多模态: 包含 visual encoder | |
| **混合部署** | 简单, 一个服务 | 配置需要折中 |
| | VPC/内部 API 够用 | 性能不是最优 |
| | | 延迟不可预测 |

```bash
# 分离部署示例

# 服务 1: 纯文本 (高并发)
vllm serve model --port 8001     --max-num-seqs 64     --max-model-len 4096

# 服务 2: Agent (长 context, prefix caching)
vllm serve model --port 8002     --max-num-seqs 16     --max-model-len 16384     --enable-prefix-caching

# 服务 3: 多模态 (含 visual encoder)
vllm serve llava-model --port 8003     --max-num-seqs 8     --limit-mm-per-prompt image=5
```

## 4. 实战: 多模态 + 工具调用混合场景

In [ ]:
# 多模态 + 工具调用的显存规划

def multimodal_memory_plan(model_gb=7, visual_encoder_gb=2, kv_dtype="fp16",
                            n_layers=32, n_kv_heads=8, head_dim=128,
                            images_per_req=3, visual_tokens_per_img=576,
                            text_tokens_per_req=2000, n_concurrent=10):
    
    bytes_per = {"fp16": 2, "fp8": 1, "q8_0": 1}
    bpe = bytes_per[kv_dtype]
    
    # Visual KV Cache
    visual_tokens = images_per_req * visual_tokens_per_img
    vis_kv_per_token = 2 * n_layers * n_kv_heads * head_dim * bpe
    vis_kv_total = visual_tokens * vis_kv_per_token * n_concurrent / (1024**3)
    
    # Text KV Cache
    text_kv_total = text_tokens_per_req * vis_kv_per_token * n_concurrent / (1024**3)
    
    # Total
    total = model_gb + visual_encoder_gb + vis_kv_total + text_kv_total + 3  # +3GB misc
    
    print(f"多模态 + 工具调用显存规划 ({n_concurrent} 并发)")
    print(f"{'─'*50}")
    print(f"  LLM 权重:              {model_gb:>6.1f} GB")
    print(f"  Visual Encoder:        {visual_encoder_gb:>6.1f} GB")
    print(f"  Visual KV Cache:       {vis_kv_total:>6.1f} GB  "
          f"({images_per_req}图×{visual_tokens_per_img}tokens×{n_concurrent}并发)")
    print(f"  Text KV Cache:         {text_kv_total:>6.1f} GB  "
          f"({text_tokens_per_req}tokens×{n_concurrent}并发)")
    print(f"  Misc (激活值+系统):     {3:>6.1f} GB")
    print(f"  {'─'*50}")
    print(f"  总计:                  {total:>6.1f} GB")
    
    if total > 80:
        print(f"  ❌ 超出 A100-80GB! 超出 {total-80:.1f} GB")
    elif total > 40:
        print(f"  ✅ 适合 A100-80GB, 不适合 A100-40GB")
    else:
        print(f"  ✅ 适合 A100-40GB")

# LLaMA-7B + LLaVA
multimodal_memory_plan(model_gb=14, n_concurrent=8, images_per_req=3)

# LLaMA-7B Q4_K_M + LLaVA (量化)
print()
multimodal_memory_plan(model_gb=4, n_concurrent=4, kv_dtype="q8_0", images_per_req=3)

# 工具调用场景
print(f"\n{'='*50}")
print(f"工具调用场景 KV Cache 演算:")
print(f"{'─'*50}")
rounds = [1, 3, 5, 10]
for r in rounds:
    tokens = 2000 + r * (20 + 30 + 1000 + 100)  # system + r*(user+tool_call+result+asst)
    kv_gb = tokens * 2 * 32 * 8 * 128 * 2 / (1024**3)
    print(f"  {r} 轮: {tokens:,} tokens → KV Cache: {kv_gb:.2f} GB/请求")
print(f"  10 并发 × 10 轮: {kv_gb*10:.1f} GB KV Cache")
print(f"  ⚠️  需要 ~30GB 仅 KV Cache → 配合 prefix caching 和量化")